In [ ]:
!pip install ucimlrepo

from ucimlrepo import fetch_ucirepo
import pandas as pd

# Fetch dataset using its unique ID
parkinsons_telemonitoring = fetch_ucirepo(id=189)

X = parkinsons_telemonitoring.data.features
y = parkinsons_telemonitoring.data.targets

df = pd.concat([X, y], axis=1)

print("Dataset Loaded Successfully!")
print(df.head())

Dataset Loaded Successfully!
   age  test_time  Jitter(%)  Jitter(Abs)  Jitter:RAP  Jitter:PPQ5  \
0   72     5.6431    0.00662     0.000034     0.00401      0.00317   
1   72    12.6660    0.00300     0.000017     0.00132      0.00150   
2   72    19.6810    0.00481     0.000025     0.00205      0.00208   
3   72    25.6470    0.00528     0.000027     0.00191      0.00264   
4   72    33.6420    0.00335     0.000020     0.00093      0.00130   

   Jitter:DDP  Shimmer  Shimmer(dB)  Shimmer:APQ3  ...  Shimmer:APQ11  \
0     0.01204  0.02565        0.230       0.01438  ...        0.01662   
1     0.00395  0.02024        0.179       0.00994  ...        0.01689   
2     0.00616  0.01675        0.181       0.00734  ...        0.01458   
3     0.00573  0.02309        0.327       0.01106  ...        0.01963   
4     0.00278  0.01703        0.176       0.00679  ...        0.01819   

   Shimmer:DDA       NHR     HNR     RPDE      DFA      PPE  sex  motor_UPDRS  \
0      0.04314  0.014290  21.6

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Split into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Scaling (Vital for SVR, Ridge, Lasso, and ElasticNet)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Features ready: {X_train_scaled.shape}")
print(f"Targets: {y_train.columns.tolist()}")

Features ready: (4700, 19)
Targets: ['motor_UPDRS', 'total_UPDRS']


In [ ]:
print(parkinsons_telemonitoring.variables)

             name     role        type demographic  \
0        subject#       ID     Integer        None   
1             age  Feature     Integer         Age   
2       test_time  Feature  Continuous        None   
3       Jitter(%)  Feature  Continuous        None   
4     Jitter(Abs)  Feature  Continuous        None   
5      Jitter:RAP  Feature  Continuous        None   
6     Jitter:PPQ5  Feature  Continuous        None   
7      Jitter:DDP  Feature  Continuous        None   
8         Shimmer  Feature  Continuous        None   
9     Shimmer(dB)  Feature  Continuous        None   
10   Shimmer:APQ3  Feature  Continuous        None   
11   Shimmer:APQ5  Feature  Continuous        None   
12  Shimmer:APQ11  Feature  Continuous        None   
13    Shimmer:DDA  Feature  Continuous        None   
14            NHR  Feature  Continuous        None   
15            HNR  Feature  Continuous        None   
16           RPDE  Feature  Continuous        None   
17            DFA  Feature  

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn import metrics

# 1. Load the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/telemonitoring/parkinsons_updrs.data"
df = pd.read_csv(url)

# 2. Preprocessing
X = df.drop(columns=['subject#', 'motor_UPDRS', 'total_UPDRS'])
y = df[['motor_UPDRS', 'total_UPDRS']]

# 3. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Helper function for sMAPE
def calculate_smape(y_true, y_pred):
    return 100/len(y_true) * np.sum(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))

# 5. Training and Evaluation Function with Scaling
def evaluate_scaled_model(X_tr, X_te, y_tr, y_te, target_name):
    # Create a pipeline that scales data then applies Linear Regression
    model_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', LinearRegression())
    ])

    model_pipeline.fit(X_tr, y_tr)
    y_pred = model_pipeline.predict(X_te)

    # Handling potential negative predictions for MSLE
    y_pred_clipped = np.clip(y_pred, 1e-6, None)

    print(f"--- Performance Metrics (SCALED) for {target_name} ---")
    print(f"MAE:                     {metrics.mean_absolute_error(y_te, y_pred):.4f}")
    print(f"MSE:                     {metrics.mean_squared_error(y_te, y_pred):.4f}")
    print(f"RMSE:                    {np.sqrt(metrics.mean_squared_error(y_te, y_pred)):.4f}")
    print(f"R^2 Score:               {metrics.r2_score(y_te, y_pred):.4f}")
    print(f"MAPE:                    {metrics.mean_absolute_percentage_error(y_te, y_pred):.4f}")
    print(f"sMAPE:                   {calculate_smape(y_te, y_pred):.2f}%")
    print(f"MSLE:                    {metrics.mean_squared_log_error(y_te, y_pred_clipped):.4f}")
    print(f"Median Absolute Error:   {metrics.median_absolute_error(y_te, y_pred):.4f}")
    print("\n")

# Run evaluations independently
evaluate_scaled_model(X_train, X_test, y_train['motor_UPDRS'], y_test['motor_UPDRS'], "Motor UPDRS")
evaluate_scaled_model(X_train, X_test, y_train['total_UPDRS'], y_test['total_UPDRS'], "Total UPDRS")

--- Performance Metrics (SCALED) for Motor UPDRS ---
MAE:                     6.3534
MSE:                     56.0142
RMSE:                    7.4843
R^2 Score:               0.1224
MAPE:                    0.3838
sMAPE:                   31.67%
MSLE:                    0.1433
Median Absolute Error:   6.1490


--- Performance Metrics (SCALED) for Total UPDRS ---
MAE:                     8.0538
MSE:                     93.3067
RMSE:                    9.6595
R^2 Score:               0.1580
MAPE:                    0.3471
sMAPE:                   29.03%
MSLE:                    0.1276
Median Absolute Error:   7.6067




In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Reusing the existing split (X_train, X_test, y_train, y_test) from previous code

def evaluate_polynomial_model(X_tr, X_te, y_tr, y_te, target_name, degree=2):
    # Create a pipeline: Scale -> Generate Polynomials -> Linear Regression
    poly_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('poly_features', PolynomialFeatures(degree=degree)),
        ('regressor', LinearRegression())
    ])

    poly_pipeline.fit(X_tr, y_tr)
    y_pred = poly_pipeline.predict(X_te)

    # Clip for MSLE
    y_pred_clipped = np.clip(y_pred, 1e-6, None)

    print(f"--- Performance Metrics (POLYNOMIAL Degree {degree}) for {target_name} ---")
    print(f"MAE:                     {metrics.mean_absolute_error(y_te, y_pred):.4f}")
    print(f"MSE:                     {metrics.mean_squared_error(y_te, y_pred):.4f}")
    print(f"RMSE:                    {np.sqrt(metrics.mean_squared_error(y_te, y_pred)):.4f}")
    print(f"R^2 Score:               {metrics.r2_score(y_te, y_pred):.4f}")
    print(f"MAPE:                    {metrics.mean_absolute_percentage_error(y_te, y_pred):.4f}")
    print(f"sMAPE:                   {calculate_smape(y_te, y_pred):.2f}%")
    print(f"MSLE:                    {metrics.mean_squared_log_error(y_te, y_pred_clipped):.4f}")
    print(f"Median Absolute Error:   {metrics.median_absolute_error(y_te, y_pred):.4f}")
    print("\n")

# Run for both targets
evaluate_polynomial_model(X_train, X_test, y_train['motor_UPDRS'], y_test['motor_UPDRS'], "Motor UPDRS")
evaluate_polynomial_model(X_train, X_test, y_train['total_UPDRS'], y_test['total_UPDRS'], "Total UPDRS")

--- Performance Metrics (POLYNOMIAL Degree 2) for Motor UPDRS ---
MAE:                     5.8564
MSE:                     99.8852
RMSE:                    9.9943
R^2 Score:               -0.5649
MAPE:                    0.3512
sMAPE:                   28.36%
MSLE:                    0.1360
Median Absolute Error:   4.9988


--- Performance Metrics (POLYNOMIAL Degree 2) for Total UPDRS ---
MAE:                     7.5284
MSE:                     142.9461
RMSE:                    11.9560
R^2 Score:               -0.2900
MAPE:                    0.3247
sMAPE:                   26.74%
MSLE:                    0.1235
Median Absolute Error:   6.2650




In [ ]:
from sklearn.linear_model import Ridge

def evaluate_ridge_model(X_tr, X_te, y_tr, y_te, target_name, alpha=1.0):
    # Pipeline: Scale -> Polynomials -> Ridge (Regularized Linear Model)
    ridge_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('poly_features', PolynomialFeatures(degree=2)),
        ('regressor', Ridge(alpha=alpha))
    ])

    ridge_pipeline.fit(X_tr, y_tr)
    y_pred = ridge_pipeline.predict(X_te)

    # Clip for MSLE
    y_pred_clipped = np.clip(y_pred, 1e-6, None)

    print(f"--- Performance Metrics (RIDGE + POLY) for {target_name} ---")
    print(f"Alpha (Penalty):         {alpha}")
    print(f"MAE:                     {metrics.mean_absolute_error(y_te, y_pred):.4f}")
    print(f"MSE:                     {metrics.mean_squared_error(y_te, y_pred):.4f}")
    print(f"RMSE:                    {np.sqrt(metrics.mean_squared_error(y_te, y_pred)):.4f}")
    print(f"R^2 Score:               {metrics.r2_score(y_te, y_pred):.4f}")
    print(f"MAPE:                    {metrics.mean_absolute_percentage_error(y_te, y_pred):.4f}")
    print(f"sMAPE:                   {calculate_smape(y_te, y_pred):.2f}%")
    print(f"MSLE:                    {metrics.mean_squared_log_error(y_te, y_pred_clipped):.4f}")
    print(f"Median Absolute Error:   {metrics.median_absolute_error(y_te, y_pred):.4f}")
    print("\n")

# Run for both targets
evaluate_ridge_model(X_train, X_test, y_train['motor_UPDRS'], y_test['motor_UPDRS'], "Motor UPDRS", alpha=10.0)
evaluate_ridge_model(X_train, X_test, y_train['total_UPDRS'], y_test['total_UPDRS'], "Total UPDRS", alpha=10.0)

--- Performance Metrics (RIDGE + POLY) for Motor UPDRS ---
Alpha (Penalty):         10.0
MAE:                     5.7198
MSE:                     70.8199
RMSE:                    8.4155
R^2 Score:               -0.1095
MAPE:                    0.3463
sMAPE:                   27.99%
MSLE:                    0.1292
Median Absolute Error:   5.0097


--- Performance Metrics (RIDGE + POLY) for Total UPDRS ---
Alpha (Penalty):         10.0
MAE:                     7.3540
MSE:                     113.1610
RMSE:                    10.6377
R^2 Score:               -0.0212
MAPE:                    0.3189
sMAPE:                   26.23%
MSLE:                    0.1162
Median Absolute Error:   6.3609




In [ ]:
from sklearn.linear_model import Lasso

def evaluate_lasso_model(X_tr, X_te, y_tr, y_te, target_name, alpha=0.1):
    # Pipeline: Scale -> Polynomials -> Lasso (L1 Regularization)
    lasso_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('poly_features', PolynomialFeatures(degree=2)),
        ('regressor', Lasso(alpha=alpha, max_iter=10000)) # Increased iterations for convergence
    ])

    lasso_pipeline.fit(X_tr, y_tr)
    y_pred = lasso_pipeline.predict(X_te)

    # Clip for MSLE
    y_pred_clipped = np.clip(y_pred, 1e-6, None)

    print(f"--- Performance Metrics (LASSO + POLY) for {target_name} ---")
    print(f"Alpha (Penalty):         {alpha}")
    print(f"MAE:                     {metrics.mean_absolute_error(y_te, y_pred):.4f}")
    print(f"MSE:                     {metrics.mean_squared_error(y_te, y_pred):.4f}")
    print(f"RMSE:                    {np.sqrt(metrics.mean_squared_error(y_te, y_pred)):.4f}")
    print(f"R^2 Score:               {metrics.r2_score(y_te, y_pred):.4f}")
    print(f"MAPE:                    {metrics.mean_absolute_percentage_error(y_te, y_pred):.4f}")
    print(f"sMAPE:                   {calculate_smape(y_te, y_pred):.2f}%")
    print(f"MSLE:                    {metrics.mean_squared_log_error(y_te, y_pred_clipped):.4f}")
    print(f"Median Absolute Error:   {metrics.median_absolute_error(y_te, y_pred):.4f}")

    # Check how many features Lasso actually used
    used_features = np.sum(lasso_pipeline.named_steps['regressor'].coef_ != 0)
    print(f"Features used by Lasso:  {used_features} out of 210")
    print("\n")

# Run for both targets
evaluate_lasso_model(X_train, X_test, y_train['motor_UPDRS'], y_test['motor_UPDRS'], "Motor UPDRS", alpha=0.1)
evaluate_lasso_model(X_train, X_test, y_train['total_UPDRS'], y_test['total_UPDRS'], "Total UPDRS", alpha=0.1)

--- Performance Metrics (LASSO + POLY) for Motor UPDRS ---
Alpha (Penalty):         0.1
MAE:                     5.8472
MSE:                     49.6674
RMSE:                    7.0475
R^2 Score:               0.2219
MAPE:                    0.3586
sMAPE:                   29.11%
MSLE:                    0.1308
Median Absolute Error:   5.3316
Features used by Lasso:  53 out of 210


--- Performance Metrics (LASSO + POLY) for Total UPDRS ---
Alpha (Penalty):         0.1
MAE:                     7.4799
MSE:                     83.1925
RMSE:                    9.1210
R^2 Score:               0.2493
MAPE:                    0.3250
sMAPE:                   26.94%
MSLE:                    0.1164
Median Absolute Error:   6.8866
Features used by Lasso:  58 out of 210




In [ ]:
from sklearn.tree import DecisionTreeRegressor

def evaluate_dt_model(X_tr, X_te, y_tr, y_te, target_name):
    # We don't strictly need a pipeline/scaler for Trees, but we can use one for consistency
    model = DecisionTreeRegressor(random_state=42, max_depth=10) # max_depth prevents extreme overfitting

    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    # Clip for MSLE
    y_pred_clipped = np.clip(y_pred, 1e-6, None)

    print(f"--- Performance Metrics (DECISION TREE) for {target_name} ---")
    print(f"MAE:                     {metrics.mean_absolute_error(y_te, y_pred):.4f}")
    print(f"MSE:                     {metrics.mean_squared_error(y_te, y_pred):.4f}")
    print(f"RMSE:                    {np.sqrt(metrics.mean_squared_error(y_te, y_pred)):.4f}")
    print(f"R^2 Score:               {metrics.r2_score(y_te, y_pred):.4f}")
    print(f"MAPE:                    {metrics.mean_absolute_percentage_error(y_te, y_pred):.4f}")
    print(f"sMAPE:                   {calculate_smape(y_te, y_pred):.2f}%")
    print(f"MSLE:                    {metrics.mean_squared_log_error(y_te, y_pred_clipped):.4f}")
    print(f"Median Absolute Error:   {metrics.median_absolute_error(y_te, y_pred):.4f}")
    print("\n")

# Run for both targets
evaluate_dt_model(X_train, X_test, y_train['motor_UPDRS'], y_test['motor_UPDRS'], "Motor UPDRS")
evaluate_dt_model(X_train, X_test, y_train['total_UPDRS'], y_test['total_UPDRS'], "Total UPDRS")

--- Performance Metrics (DECISION TREE) for Motor UPDRS ---
MAE:                     1.5825
MSE:                     9.4144
RMSE:                    3.0683
R^2 Score:               0.8525
MAPE:                    0.0815
sMAPE:                   7.84%
MSLE:                    0.0203
Median Absolute Error:   0.5648


--- Performance Metrics (DECISION TREE) for Total UPDRS ---
MAE:                     1.4172
MSE:                     11.3655
RMSE:                    3.3713
R^2 Score:               0.8974
MAPE:                    0.0637
sMAPE:                   6.23%
MSLE:                    0.0253
Median Absolute Error:   0.4515




In [ ]:
from sklearn.svm import SVR

def evaluate_svr_model(X_tr, X_te, y_tr, y_te, target_name):
    # Pipeline: Scale -> SVR (with RBF kernel for non-linearity)
    svr_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', SVR(kernel='rbf', C=100, epsilon=0.1))
    ])

    svr_pipeline.fit(X_tr, y_tr)
    y_pred = svr_pipeline.predict(X_te)

    y_pred_clipped = np.clip(y_pred, 1e-6, None)

    print(f"--- Performance Metrics (SVR) for {target_name} ---")
    print(f"MAE:                     {metrics.mean_absolute_error(y_te, y_pred):.4f}")
    print(f"MSE:                     {metrics.mean_squared_error(y_te, y_pred):.4f}")
    print(f"RMSE:                    {np.sqrt(metrics.mean_squared_error(y_te, y_pred)):.4f}")
    print(f"R^2 Score:               {metrics.r2_score(y_te, y_pred):.4f}")
    print(f"sMAPE:                   {calculate_smape(y_te, y_pred):.2f}%")
    print(f"Median Absolute Error:   {metrics.median_absolute_error(y_te, y_pred):.4f}")
    print("\n")

# Run for both
evaluate_svr_model(X_train, X_test, y_train['motor_UPDRS'], y_test['motor_UPDRS'], "Motor UPDRS")
evaluate_svr_model(X_train, X_test, y_train['total_UPDRS'], y_test['total_UPDRS'], "Total UPDRS")

--- Performance Metrics (SVR) for Motor UPDRS ---
MAE:                     3.6499
MSE:                     24.7067
RMSE:                    4.9706
R^2 Score:               0.6129
sMAPE:                   18.94%
Median Absolute Error:   2.5181


--- Performance Metrics (SVR) for Total UPDRS ---
MAE:                     4.8110
MSE:                     45.5796
RMSE:                    6.7513
R^2 Score:               0.5887
sMAPE:                   17.84%
Median Absolute Error:   3.1776




In [ ]:
from sklearn.linear_model import ElasticNet

def evaluate_elasticnet_full(X_tr, X_te, y_tr, y_te, target_name):
    # Pipeline: Scale -> ElasticNet
    en_pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', ElasticNet(alpha=0.1, l1_ratio=0.5))
    ])

    en_pipeline.fit(X_tr, y_tr)
    y_pred = en_pipeline.predict(X_te)

    # Handling potential negative predictions for MSLE
    y_pred_clipped = np.clip(y_pred, 1e-6, None)

    print(f"--- Performance Metrics (ELASTICNET) for {target_name} ---")
    print(f"MAE:                     {metrics.mean_absolute_error(y_te, y_pred):.4f}")
    print(f"MSE:                     {metrics.mean_squared_error(y_te, y_pred):.4f}")
    print(f"RMSE:                    {np.sqrt(metrics.mean_squared_error(y_te, y_pred)):.4f}")
    print(f"R^2 Score:               {metrics.r2_score(y_te, y_pred):.4f}")
    print(f"MAPE:                    {metrics.mean_absolute_percentage_error(y_te, y_pred):.4f}")
    print(f"sMAPE:                   {calculate_smape(y_te, y_pred):.2f}%")
    print(f"MSLE:                    {metrics.mean_squared_log_error(y_te, y_pred_clipped):.4f}")
    print(f"Median Absolute Error:   {metrics.median_absolute_error(y_te, y_pred):.4f}")
    print("\n")

# Run for both targets
evaluate_elasticnet_full(X_train, X_test, y_train['motor_UPDRS'], y_test['motor_UPDRS'], "Motor UPDRS")
evaluate_elasticnet_full(X_train, X_test, y_train['total_UPDRS'], y_test['total_UPDRS'], "Total UPDRS")

--- Performance Metrics (ELASTICNET) for Motor UPDRS ---
MAE:                     6.3930
MSE:                     56.0081
RMSE:                    7.4839
R^2 Score:               0.1225
MAPE:                    0.3890
sMAPE:                   31.93%
MSLE:                    0.1448
Median Absolute Error:   6.2692


--- Performance Metrics (ELASTICNET) for Total UPDRS ---
MAE:                     8.0506
MSE:                     93.1102
RMSE:                    9.6494
R^2 Score:               0.1598
MAPE:                    0.3496
sMAPE:                   29.09%
MSLE:                    0.1284
Median Absolute Error:   7.6769




In [ ]:
from sklearn.multioutput import RegressorChain
from sklearn.ensemble import RandomForestRegressor

def run_sophisticated_chain_full(X_tr, X_te, y_tr, y_te):
    # Setup the Chain with Random Forest
    base_model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42)
    chain = RegressorChain(base_estimator=base_model, order=[0, 1])

    chain.fit(X_tr, y_tr)
    y_pred = chain.predict(X_te)

    # Loop through both targets to print metrics
    targets = ["Motor UPDRS", "Total UPDRS"]

    for i, target_name in enumerate(targets):
        y_true = y_te.iloc[:, i]
        y_p = y_pred[:, i]

        # Clip for MSLE
        y_p_clipped = np.clip(y_p, 1e-6, None)

        print(f"--- REGRESSOR CHAIN: {target_name} ---")
        print(f"MAE:                     {metrics.mean_absolute_error(y_true, y_p):.4f}")
        print(f"MSE:                     {metrics.mean_squared_error(y_true, y_p):.4f}")
        print(f"RMSE:                    {np.sqrt(metrics.mean_squared_error(y_true, y_p)):.4f}")
        print(f"R^2 Score:               {metrics.r2_score(y_true, y_p):.4f}")
        print(f"MAPE:                    {metrics.mean_absolute_percentage_error(y_true, y_p):.4f}")
        print(f"sMAPE:                   {calculate_smape(y_true, y_p):.2f}%")
        print(f"MSLE:                    {metrics.mean_squared_log_error(y_true, y_p_clipped):.4f}")
        print(f"Median Absolute Error:   {metrics.median_absolute_error(y_true, y_p):.4f}")
        print("\n")

# Run the full evaluation
run_sophisticated_chain_full(X_train, X_test, y_train, y_test)

--- REGRESSOR CHAIN: Motor UPDRS ---
MAE:                     0.8694
MSE:                     3.1327
RMSE:                    1.7699
R^2 Score:               0.9509
MAPE:                    0.0453
sMAPE:                   4.41%
MSLE:                    0.0071
Median Absolute Error:   0.2557


--- REGRESSOR CHAIN: Total UPDRS ---
MAE:                     1.1678
MSE:                     5.8174
RMSE:                    2.4119
R^2 Score:               0.9475
MAPE:                    0.0492
sMAPE:                   4.61%
MSLE:                    0.0095
Median Absolute Error:   0.3917




**Time-aware prediction**

In [ ]:
# Rename columns (optional but useful)
df.columns = df.columns.str.strip()

# Sort data by patient and time
df = df.sort_values(by=['subject#', 'test_time'])

print(df.head())

    subject#  age  sex  test_time  motor_UPDRS  total_UPDRS  Jitter(%)  \
0          1   72    0     5.6431       28.199       34.398    0.00662   
24         1   72    0     5.6431       28.199       34.398    0.00348   
49         1   72    0     5.6438       28.199       34.398    0.00413   
74         1   72    0     5.6451       28.199       34.398    0.00217   
99         1   72    0     5.6458       28.199       34.399    0.00294   

    Jitter(Abs)  Jitter:RAP  Jitter:PPQ5  ...  Shimmer(dB)  Shimmer:APQ3  \
0      0.000034     0.00401      0.00317  ...        0.230       0.01438   
24     0.000015     0.00124      0.00133  ...        0.113       0.00411   
49     0.000021     0.00173      0.00165  ...        0.125       0.00555   
74     0.000011     0.00086      0.00099  ...        0.072       0.00271   
99     0.000015     0.00121      0.00125  ...        0.098       0.00459   

    Shimmer:APQ5  Shimmer:APQ11  Shimmer:DDA       NHR     HNR     RPDE  \
0        0.01309       

In [ ]:
# Create lag features for target
df['motor_UPDRS_lag1'] = df.groupby('subject#')['motor_UPDRS'].shift(1)
df['motor_UPDRS_lag2'] = df.groupby('subject#')['motor_UPDRS'].shift(2)

# Create lag for some important voice features
df['Jitter_lag1'] = df.groupby('subject#')['Jitter(%)'].shift(1)
df['Shimmer_lag1'] = df.groupby('subject#')['Shimmer'].shift(1)
df['HNR_lag1'] = df.groupby('subject#')['HNR'].shift(1)

In [ ]:
df = df.dropna()

print("After lagging:", df.shape)

After lagging: (5791, 27)


In [ ]:
# Target
y = df['motor_UPDRS']

# Drop unwanted columns
X = df.drop(columns=[
    'motor_UPDRS',
    'total_UPDRS',
    'subject#'
])

In [ ]:
# Choose a time threshold (you can adjust)
threshold = df['test_time'].quantile(0.8)

train_df = df[df['test_time'] < threshold]
test_df = df[df['test_time'] >= threshold]

X_train = train_df[X.columns]
y_train = train_df['motor_UPDRS']

X_test = test_df[X.columns]
y_test = test_df['motor_UPDRS']

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (4627, 24)
Test size: (1164, 24)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)

model.fit(X_train_scaled, y_train)

RandomForestRegressor(random_state=42)

In [ ]:
y_pred = model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 0.18637948281786892
RMSE: 0.3160220424611601
R2 Score: 0.9983978943802603


**Only voice features (no target lag)**

In [ ]:
# Drop target lag columns (VERY IMPORTANT)
df = df.drop(columns=[
    'motor_UPDRS_lag1',
    'motor_UPDRS_lag2'
])

In [ ]:
# Target
y = df['motor_UPDRS']

# Features (NO subject#, NO target, NO target lag)
X = df.drop(columns=[
    'motor_UPDRS',
    'total_UPDRS',
    'subject#'
])

In [ ]:
train_list = []
test_list = []

for subject in df['subject#'].unique():
    sub_df = df[df['subject#'] == subject]

    split_index = int(0.8 * len(sub_df))

    train_list.append(sub_df.iloc[:split_index])
    test_list.append(sub_df.iloc[split_index:])

train_df = pd.concat(train_list)
test_df = pd.concat(test_list)

X_train = train_df[X.columns]
y_train = train_df['motor_UPDRS']

X_test = test_df[X.columns]
y_test = test_df['motor_UPDRS']

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

RandomForestRegressor(random_state=42)

In [ ]:
y_pred = model.predict(X_test_scaled)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 1.8536065824829908
RMSE: 2.710427477308781
R2 Score: 0.8945826207494659


We explored three approaches for predicting Parkinson’s disease severity.

First, we treated the dataset as a standard regression problem, ignoring temporal dependencies. While this provided a good baseline, it failed to capture the progression of the disease.

Next, we introduced lag features, allowing the model to use past severity scores to predict future values. This resulted in very high accuracy, but it relied heavily on temporal continuity, making it less meaningful for real-world prediction.

Finally, we removed the lag features to ensure the model depended only on voice-based biomarkers. This produced slightly lower but more realistic performance, demonstrating that voice characteristics can effectively predict disease severity.

This comparison highlights the importance of handling time-series data carefully and avoiding data leakage in machine learning models.
